# A Dispensing Calculator for a Small Formulary

In this exercise, we will build a program that a hospital pharmacy could actually use.
You start from a table of [atomic masses](https://en.wikipedia.org/wiki/Atomic_mass), turn molecular formulas into molecular weights, store a whole formulary as nested dictionaries, work out how many tablets a patient of a given body weight needs, and finish by summarising the formulary with comprehensions and sets.

Everything is one program.
What we build in the beginning (**Part 1**) is used later when we are finally done (**Part 4** and **Part 5**).

By the end you will have printed a dosing report like this one:

```
sulfamethoxazole   required 1480 mg   3 tablet(s) of 400 mg   leftover 280 mg
```


## Part 1 - From a formula to a molecular weight

A [molecular formula](https://en.wikipedia.org/wiki/Chemical_formula#Molecular_formula) is nothing but a mapping from an element symbol to an atom count, so a [Python dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries) represents it exactly:
[Paracetamol](https://en.wikipedia.org/wiki/Paracetamol) has the molecular formula $\mathrm{C_8H_9NO_2}$ which we can translate into a dictionary `{"C": 8, "H": 9, "N": 1, "O": 2}`.

To turn that into a [molecular weight](https://en.wikipedia.org/wiki/Molar_mass) we need a second dictionary — a lookup table of **average** atomic masses:

| symbol | atomic mass (g/mol) |
|---|---|
| H | 1.008 |
| C | 12.011 |
| N | 14.007 |
| O | 15.999 |
| S | 32.06 |

The molecular weight $M$ is expressed as the formula

$$M = \sum_{\text{elements } i} n_i \, m_i$$

where $n_i$ is the number of atoms of element $i$ and $m_i$ the atomic mass of element $i$


### Questions
  1. Discuss with your neighbour (or yourself) if the information above is enough to use the formula to calculate $M$.

### Questions
  2. Below a new dictionary (`atomic_masses`) is created with atomic masses for some elements. Use the values from the table above to insert the missing values at the `_`.
  3. Print the dictionary and confirm that it has 5 items using the `len` function.
  4. Use the `atomic_masses` table you just defined and the molecular formula for paracetamol to compute the molecular mass of paracetamol. Compare with literature values.
  5. Discuss what the `round(mw_paracetamol, 2)` function call does.

In [ ]:
atomic_masses = {"H": 1.008, "C": _, "N": _, "O": 15.999, "S": _}
print(_)
print("elements in the table:", len(_))

In [ ]:
mw_paracetamol = 8 * atomic_masses["C"] + _ * atomic_masses["H"] + _  #! the molecular formula is C8H9NO2, one lookup per element
print("paracetamol:", round(mw_paracetamol, 2), "g/mol")

Writing that sum out by hand for *every* compound would be unbearable, so here are two functions you may use for the rest of the session (and later in the course):

In [ ]:
def molecular_weight(formula: dict[str, int], masses: dict[str, float]) -> float:
    """Return the molecular weight in g/mol of a formula dictionary such as {"C": 8, "H": 9}."""
    total = 0.0
    for element, count in formula.items():
        total = total + count * masses[element]
    return total

def mmol_from_mg(mass_mg: float, mw: float) -> float:
    """Return the amount in millimoles of mass_mg milligrams of a compound of molar mass mw."""
    return mass_mg / mw

### Questions
  6. When you run the above code block, nothing happens. Why?
  7. Explain in your own words what the above functions do.

Now we will use the helper functions and compute the molecular weight more easily.
Below is paracetamol shown

In [ ]:
paracetamol_formula = {"C": 8, "H": 9, "N": 1, "O": 2}
mw_paracetamol = molecular_weight(paracetamol_formula, atomic_masses)
print("by function:", round(mw_paracetamol, 2))

### Question
  8. Do it also for aspirin and caffeine.

Caffeine comes out at 194.19 g/mol, which is the literature value - a good sign that both the formula and the table are right.
Always check one compound you know before trusting a calculation on ninety others you do not.

[Loratadine](https://en.wikipedia.org/wiki/Loratadine) is a compound that is used to treat allergies.

### Question
  9. Explain what happens when you try and use our code to obtain the molecular mass of loratadine.
  10. Fix the problem so it computes the correct molecular weight of 382.89 g/mol.

In [ ]:
loratadine_formula = {"C": 22, _}  # update the formula
mw_loratadine = molecular_weight(loratadine_formula, atomic_masses)
print("loratadine:", round(_, 2), "g/mol")

## Part 2 - The formulary as a nested dictionary

A pharmacy does not hold one compound, it holds a formulary. Each drug has a record, and each record is itself a dictionary, i.e., a name pointing to a dictionary of properties, one of whose values is again a dictionary (the formula). Two sets of square brackets reach into it: `formulary["aspirin"]["dose_mg"]`.

The five starting records are common oral drugs:

| drug | formula | tablet strength (mg) | mg per kg body weight | max single dose (mg) |
|---|---|---|---|---|
| paracetamol | $\mathrm{C_8H_9NO_2}$ | 500 | 15 | 1000 |
| aspirin | $\mathrm{C_9H_8O_4}$ | 500 | 10 | 1000 |
| ibuprofen | $\mathrm{C_{13}H_{18}O_2}$ | 400 | 7 | 800 |
| caffeine | $\mathrm{C_8H_{10}N_4O_2}$ | 100 | 2 | 200 |
| sulfamethoxazole | $\mathrm{C_{10}H_{11}N_3O_3S}$ | 400 | 20 | 1600 |

The `mg_per_kg` column is what makes the dose patient-specific, and it is what Part 3 uses below.

The `formulary` is presented below for you.


In [ ]:
formulary = {
    "paracetamol": {"dose_mg": 500, "mg_per_kg": 15, "max_single_mg": 1000, "formula": {"C": 8, "H": 9, "N": 1, "O": 2}},
    "aspirin": {"dose_mg": 500, "mg_per_kg": 10, "max_single_mg": 1000, "formula": {"C": 9, "H": 8, "O": 4}},
    "ibuprofen": {"dose_mg": 400, "mg_per_kg": 7, "max_single_mg": 800, "formula": {"C": 13, "H": 18, "O": 2}},
    "caffeine": {"dose_mg": 100, "mg_per_kg": 2, "max_single_mg": 200, "formula": {"C": 8, "H": 10, "N": 4, "O": 2}},
    "sulfamethoxazole": {"dose_mg": 400, "mg_per_kg": 20, "max_single_mg": 1600, "formula": {"C": 10, "H": 11, "N": 3, "O": 3, "S": 1}},
}
print("drugs in the formulary:", len(formulary))

### Questions
  1. print the entire record for caffeine.
  2. Print the tablet strength of ibuprofen.
  3. Print the number of sulfur atoms, $n_S$, in sulfamethoxazole.

In [ ]:
print("caffeine record:", formulary["_"])
print(formulary[_]["dose_mg"], "mg per tablet")
print(formulary[_][_]["S"], "sulfur atom(s)")

Metformin, $\mathrm{C_4H_{11}N_5}$, is added to the formulary: 500 mg tablets, 10 mg/kg, maximum single dose 1000 mg. Assigning to a new key that does not exist yet creates the entry as shown below:


In [ ]:
formulary["metformin"] = {"dose_mg": 500, "mg_per_kg": 10, "max_single_mg": 1000, "formula": {"C": 4, "H": 11, "N": 5}}  #! add the metformin record

### Questions
  4. Add your favorite drug-like molecule to your formulary.

After you have added your favorite drug, you can print the enture formulary using the code below.

In [ ]:
for name, record in formulary.items():
    mw = molecular_weight(record["formula"], atomic_masses)  #! molecular weight of this record's formula
    mmol = mmol_from_mg(record["dose_mg"], mw)  #! millimoles of active ingredient in one tablet
    print(f"{name:16s}: MW {round(mw, 2)} g/mol, one tablet = {round(mmol, 3)} mmol")

Look at the two 500 mg tablets in the above printout.
Paracetamol and metformin weigh the same on the balance, but a metformin tablet contains about 3.9 mmol and a paracetamol tablet about 3.3 mmol, because metformin is the lighter molecule.
Milligrams are what a pharmacist dispenses.
Millimoles are what the receptor sees.
Any time you compare two different drugs, ask yourself which of the two units the comparison should be in.

That table is the formulary described in the introduction.
What a pharmacy actually needs is the dose for the patient standing at the counter - so next we bring in body weight of a patient.

### Questions

5. Sulfamethoxazole is the only compound here containing sulfur. What exactly would `molecular_weight` do if `"S"` was missing from `atomic_masses`?
6. The formulary is a dictionary keyed by drug name. What would you lose, in practice, if it was a `list` of records instead if a dictionary?
7. Metformin and paracetamol tablets both weigh 500 mg but differ in millimoles. Which of the two numbers would you put on a comparison of potency, and why?


## Part 3 - Weight-based dosing and whole tablets

A dose is prescribed per kilogram of body weight, but never above a fixed ceiling, because above a certain amount the extra drug only adds toxicity:

$$\text{dose} = \min\left(\text{mg per kg} \times \text{body weight},\ \text{max single dose}\right)$$

Then comes the practical problem. Tablets do not divide. If the patient needs 518 mg of ibuprofen and the tablet is 400 mg, they get **one** whole tablet and 118 mg of the requirement is not met. Those are exactly floor division and modulo you saw in the videos and preparational exercises.

```py
tablets = required_mg // strength_mg
leftover_mg = required_mg % strength_mg
```

Our patient today weighs 74 kg.


### Questions
  1. In python, one can use the `min` function. It takes two arguments. Guess the output of the following statements before trying yourself: `min(3,5)`, `min(5,3)`, `min(2,2)`.

In [ ]:
print(_, _, _)

Below is a function that calculates a single dose for a patient given three different values as input: `body_kg` is the body mass in kilograms of the patient, `mg_per_kg` is the weight-based dose and `max_single_mg` is the maximum amount a patient can take in in one dose.

In [ ]:
def per_dose_mg(body_kg: float, mg_per_kg: float, max_single_mg: float) -> float:
    """Return the single dose in mg for this patient, capped at the maximum single dose."""
    dose_mg = body_kg * mg_per_kg  #! the uncapped weight-based dose
    if dose_mg > max_single_mg:  #! apply the ceiling and return it early
        return max_single_mg  #!
    return dose_mg  #! otherwise the weight-based dose

### Questions
  2. Explain what the function `per_dose_mg` does and the possible values it can output.

Below, two additional functions are given that implements two concepts described above.

### Question
  3. Explain the functions and what they do. What `type` is the output?

In [ ]:
def tablets_needed(required_mg: float, strength_mg: float) -> int:
    """Return the number of whole tablets that fit inside the required dose."""
    return required_mg // strength_mg  #! whole tablets only

def leftover_mg(required_mg: float, strength_mg: float) -> float:
    """Return the milligrams of the requirement not covered by whole tablets."""
    return required_mg % strength_mg  #! the remainder

### Questions
  4. What is the output of the following code without running it? (run it when you have guessed)

In [ ]:
print(tablets_needed(518, 400), "tablet(s), leftover mass:", leftover_mg(518, 400), "mg")

Now we return to our patient.
The patients mass was defined to be 74 kg.
We now focus on how each drug in our formulary suits this particular patient.

### Questions
  5. fix the missing parts in the code below so we can evaluate the drugs for our patient. *hint: remember that we used loops in Part 2 above. Look there to see where you can find the data you need.*

In [ ]:
body_mass_kg = _

for name, record in formulary.items():
    required_mg = per_dose_mg(body_mass_kg, record["mg_per_kg"], record["max_single_mg"])
    n_tablets = tablets_needed(required_mg, record["_"])  #! whole tablets of this strength
    missing_mg = leftover_mg(_, _)  #! milligrams left uncovered
    print(f"{name}: required {required_mg} mg, {n_tablets} tablet(s) of {record['dose_mg']} mg, leftover {missing_mg} mg")

### Questions
  6. Paracetamol has a leftover of 0 mg. What does that mean?

This of course prints *all* drugs in the formulary. However, suppose the pharmacy wants to be alerted when one drug reaches has too much spill (leverover).
This can be achived by evaluating the `missing_mg` variable inside the `for` loop and using a `break` statement to exit the loop as soon as condition is met.

### Questions
  7. update the code below to stop when the first drug with a `missing_mg` amount is above 100 mg.
  8. Print the `name` and the `missing_mg` variable with a helpful message.

In [ ]:
for name, record in formulary.items():  #! scan the formulary and stop at the first drug with a leftover above 100 mg
    required_mg = per_dose_mg(body_mass_kg, record["mg_per_kg"], record["max_single_mg"])
    missing_mg = leftover_mg(_, record["dose_mg"])  #!
    if _ > _:
        print("suspension needed for:", name, "- leftover", missing_mg, "mg")
        break

### Additional Questions

  9. Why is `//` the right operator for the tablet count rather than `/`? What would `518 / 400` have printed, and what would a pharmacist do with that number?
  10. The `break` loop reports one drug only. If metformin had been added to the formulary *before* sulfamethoxazole, would the answer change? What does that tell you about relying on a search that stops early?
  11. For paracetamol the cap is applied: the patient needs $15 \times 74 = 1110$ mg but receives 1000 mg. Is the dose the patient actually receives still proportional to body weight? Is that a problem?


## Part 4 - Summarising the Formulary

The report for all our drugs in the formulary is a print of one drug per line.
Any pharmacy, however, would also want answers for the entire formulary: *how heavy* are these molecules, *which of them* are the large ones, *which elements* does the whole formulary draw on, and how much active ingredient does this patient swallow in one round of doses.


In python, the easiest way to do this is using [list comprehensions](https://docs.python.org/3.10/tutorial/datastructures.html?highlight=list%20comprehension#list-comprehensions) explained in the videos to make more general statements about the molecules.

### Questions
  1. We will use the `formulary.items()` function and `formulary.values()` function. Explain the difference between the two. *hint: see the [official documentation](https://docs.python.org/3/tutorial/datastructures.html#looping-techniques) for details if you can't remember.*
  2. Explain the output of the code below and compare with the input of the `formulary` that we defined in part 2 of this exercise.

In [ ]:
formulas = [record["formula"] for record in formulary.values()]
print(formulas)

When we make list comprehensions, we can modify what is put into the new list by calling other functions.
### Questions
  3. Explain the output of the code below.

In [ ]:
mw_list = [round(molecular_weight(record["formula"], atomic_masses), 2) for record in formulary.values()]  #! one MW per drug, rounded to 2 decimals
print("molecular weights:", mw_list)

### Questions
  4. Make a list comprehension with the code below to make a list of names of the molecules so we can compare with the molecular weights.

In [ ]:
names = [_ for name, record in formulary.items()]
print(names)

A formulary is also quite useful to select for certain drugs by specifying that some properties should be constrained.
For example, we might want to look for drugs with a high molecular weight.
This is also possible through list comprehensions.

### Questions
  5. Make a list comprehension stores the names (in `heavy_names`) where only molecules with a molecular weight over 180 g/mol is kept.

In [ ]:
heavy_names = [name for name, record in formulary.items() if molecular_weight(record["formula"], atomic_masses) > 180]  #! names of drugs above 180 g/mol
print("above 180 g/mol:", heavy_names)

### Questions
  6. Implement your own filter for the `"mg_per_kg"` data, but print only the names that have a low value (`<= 10`).

In [ ]:
low_dose_names = [name for name, record in formulary.items() if record["mg_per_kg"] <= 10]
print(low_dose_names)

### Questions

7. `mw_list` and `heavy_names` both loop over the formulary, but one uses `.values()` and the other `.items()`. Why can `heavy_names` not be written with `.values()`?
8. Part 3's dosing report prints three quantities per drug and formats a whole sentence. Would you rewrite it as a comprehension? Justify your answer with the checklist from video 6.
9. `total_mg` used round brackets, so nothing was stored. Suppose you also wanted the largest single dose in the formulary. Could you reuse the same generator object for `max()` after using it in `sum()`?


## Part 5 - Optional: loratadine, chlorine, and percent composition

You may stop here, or finish this at home. Nothing above depends on it.

Loratadine, the non-sedating antihistamine, is $\mathrm{C_{22}H_{23}ClN_2O_2}$ - a chlorine-containing molecule with a 10 mg tablet, 0.15 mg/kg, maximum single dose 10 mg. Adding it exposes the gap in `atomic_masses` you found in Part 1.

An elemental analysis laboratory reports composition as mass percentages,

$$\%\,i = 100 \times \frac{n_i\, m_i}{M}$$

and a synthesised batch is accepted when the *measured* carbon percentage, i.e., $\%\,C$ is within 0.4 percentage points of the *calculated* value.

In one experiment, you measure that $\%\,C = 62.9\,\%$.
### Questions
  1. Look back in the exercise, and explain $n_i$, $m_i$ and $M$.
  2. Add loratadine to the formulary by fixing the code below (dose is 10 mg, mg per kg weight is 0.15, the maximum single dose is 10 mg and the formula you can find further up).
  3. Compute the molecular mass $M$ of loratadine using your formulary and how we did it above.
  4. Compute the elemental analysis for carbon, $\%\,C$, and compare with the measured value listed above.
  5. Make the comparison using an `if`-`then`-`else` statement so that it either prints that the sample is not pure enough to use based on your comparison, or that it is pure and can be used.
  6. Discuss the result.

In [ ]:
formulary["loratadine"] = _ {"dose_mg": _, "mg_per_kg": _, "max_single_mg": _, "formula": _}
mw_loratadine = molecular_weight(formulary["loratadine"]["formula"], _)
print("loratadine:", round(mw_loratadine, 2), "g/mol")

In [ ]:
measured_C_pct = 62.9
composition_C = _ # implement the formula above
difference = abs(measured_C_pct - composition_C)  # how far the measurement is from theory

if _:
    print("batch accepted, difference", round(difference, 2), "percentage points")  #!
else:
    print("batch rejected, difference", round(difference, 2), "percentage points")  #!